# Mock Data Vector with Optical Selection Effects

Forward model for the Buzzard-like DES Y3 cluster mock including:

1. Intrinsic mass–richness relation from Costanzi et al. 2026, Eq. 15
   (DES-Y1 NC+3x2pt best fit).
2. Projection of $\lambda_\mathrm{true}\to\lambda_\mathrm{obs}$ via the
   Costanzi $P(\lambda_\mathrm{obs}|\lambda_\mathrm{true}, z)$ mixture
   (delta + positive exponential).
3. Per-halo $B_\mathrm{sel}$ weight (one-halo / small-scale limit, App. C).

Outputs:

- $N(\lambda_\mathrm{obs}, z)$ binned number counts.
- $\gamma_\mathrm{obs}(R, \lambda_\mathrm{obs}\text{-bin}, z\text{-bin}) = \text{stack}(B_\mathrm{sel}\,\gamma_\mathrm{mock})$.

Boost-factor contamination is not yet included.

In [ ]:
import os, fitsio
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, join
from astropy.cosmology import FlatLambdaCDM

import matplotlib
matplotlib.rc('xtick', labelsize=16, top=True, direction='in')
matplotlib.rc('ytick', labelsize=16, right=True, direction='in')
matplotlib.rc('axes', linewidth=1, labelsize=16)

## 1. Inputs

Everything the user can tune lives here.

In [ ]:
# --- Cosmology --------------------------------------------------------
cosmo0 = FlatLambdaCDM(H0=70, Om0=0.3, Tcmb0=2.725)

# --- Mass-richness / HOD parameters (Costanzi Eq. 15, DES-Y1 best fit).
# These are hard-coded inside costanzi_selection.py.
from costanzi_selection import (
    M_MIN, ALPHA, M_1, M_PIVOT, SIGM_INTR, EPSI, PIVOT_Z0,
    l_sat, l_tr, sig_intr, pltr_M,
    sample_lambda_true, sample_lambda_obs, projection_params,
    b_sel_one_halo,
    load_prj_interpolators, load_prj_posterior_mean,
)
print('DES-Y1 params: log10 M_min=%.4f  alpha=%.4f  log10 M_1=%.4f  sig_intr=%.4f  eps=%.4f  z_piv=%.4f'
      % (np.log10(M_MIN), ALPHA, np.log10(M_1), SIGM_INTR, EPSI, PIVOT_Z0))

# --- Projection-param file: posterior-mean collapse.
PRJ_FILE = os.path.join(os.getcwd(), 'prj_params_DESY3_lss_lin_dep_getdist_v1.txt')
PRJ_INTERP = load_prj_posterior_mean(PRJ_FILE)

# --- Selection + binning (match createDataVector.ipynb edges) --------
ZMIN, ZMAX, LOGM_MIN = 0.2, 0.65, 13.0
LBDBINS  = np.array([5, 15, 25, 40, 160])
ZMIN_LIST = np.array([0.20, 0.37, 0.51])
ZMAX_LIST = np.array([0.32, 0.51, 0.64])

RNG = np.random.default_rng(seed=42)

## 2. Load halo catalogue and $\Delta\Sigma$ profiles

Same selection as `0-MakeMock.ipynb`: `pid == -1`, $0 \le \cos i \le 1$,
drop the $0.33 \le z \le 0.37$ simulation-box seam, then require
$0.2 \le z \le 0.65$ and $\log M_\mathrm{vir} \ge 13$.

In [ ]:
from fileLoc import FileLocs
floc = FileLocs(machine='nersc')

data_h, _ = fitsio.read(floc.halo_run_fname, header=True)
data,   _ = fitsio.read(floc.profile_output_fname, header=True)

select_good = (
    (data['pid'] == -1)
    & (data['cosi'] >= 0) & (data['cosi'] <= 1)
    & ((data['redshift'] < 0.33) | (data['redshift'] > 0.37))
)

redshift_sg = data['redshift'][select_good]
logMvir_sg  = np.log10(data['Mvir'])[select_good]

sel = (redshift_sg >= ZMIN) & (redshift_sg <= ZMAX) & (logMvir_sg >= LOGM_MIN)

_mock   = Table(data[select_good][sel])
_data_h = Table(data_h); _data_h.rename_column('HALOID', 'haloid')
mock    = join(_mock, _data_h)
print('Number of halos in mock: %i' % len(mock))

data = data_h = _mock = _data_h = None

## 3. $\lambda_\mathrm{true}$ from Costanzi Eq. 15

$\langle\lambda_\mathrm{true}|M,z\rangle = 1 + \left(\tfrac{M-M_\mathrm{min}}{M_\mathrm{pivot}}\right)^{\alpha}\,\left(\tfrac{1+z}{1+z_\mathrm{piv}}\right)^{\epsilon}$,

with intrinsic scatter $\sigma_\mathrm{intr}\,\ell_\mathrm{sat}(M,z)$ and
$P(\lambda_\mathrm{true}|M,z)$ a compound-Poisson+lognormal law (`pltr_M`).

In [ ]:
Mvir = np.asarray(mock['Mvir'], dtype=float)
zcl  = np.asarray(mock['redshift'], dtype=float)

lambda_true = sample_lambda_true(Mvir, zcl, rng=RNG).astype(float)
mock['lambda_true'] = lambda_true

print('mean lambda_true:', lambda_true.mean())
print('fraction lambda_true >= 5 :', (lambda_true >= 5).mean())
print('fraction lambda_true >= 20:', (lambda_true >= 20).mean())

# Sanity: expected mean relation vs sampled halos.
logM = np.log10(Mvir)
fig, ax = plt.subplots(figsize=(6, 5))
ax.hexbin(logM, np.log10(np.clip(lambda_true, 1, None)), gridsize=60,
          mincnt=1, cmap='Greys')
mg = np.linspace(logM.min(), logM.max(), 80)
for zref in [0.25, 0.45, 0.60]:
    ax.plot(mg, np.log10(l_tr(10**mg, zref)), lw=2, label='z=%.2f' % zref)
ax.set_xlabel(r'$\log_{10} M_\mathrm{vir}\,[M_\odot]$')
ax.set_ylabel(r'$\log_{10} \lambda_\mathrm{true}$')
ax.legend(fontsize=11); fig.tight_layout()

## 4. Project $\lambda_\mathrm{true}\to\lambda_\mathrm{obs}$

$P(\lambda_\mathrm{obs}|\lambda_\mathrm{true},z) = (1-f_\mathrm{prj})\,
\delta_D(\lambda_\mathrm{obs}-\lambda_\mathrm{true}) +
f_\mathrm{prj}\,\tau\,e^{-\tau(\lambda_\mathrm{obs}-\lambda_\mathrm{true})}
\Theta_H(\lambda_\mathrm{obs}-\lambda_\mathrm{true})$.

$(f_\mathrm{prj}, \tau)(\lambda_\mathrm{true}, z)$ uses the Costanzi
`_fprj_model` / `_tau_model` forms with posterior-mean coefficients.

In [ ]:
lambda_obs, f_prj, tau = sample_lambda_obs(lambda_true, zcl, PRJ_INTERP,
                                           rng=RNG)
mock['lambda_obs'] = lambda_obs
mock['f_prj']      = f_prj
mock['tau_prj']    = tau

# Projection conservation: no halos added or removed.
assert len(lambda_obs) == len(lambda_true)

dlam = lambda_obs - lambda_true
print('<Delta lambda>        = %.3f' % dlam.mean())
print('<f_prj / tau> (pred)  = %.3f' % np.mean(f_prj / tau))
print('fraction with boost   = %.3f' % (dlam > 0).mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hexbin(np.log10(np.clip(lambda_true, 1, None)),
                np.log10(np.clip(lambda_obs, 1, None)),
                gridsize=60, mincnt=1, cmap='viridis')
axes[0].plot([-0.5, 2.5], [-0.5, 2.5], 'r--', lw=1)
axes[0].set_xlabel(r'$\log_{10}\lambda_\mathrm{true}$')
axes[0].set_ylabel(r'$\log_{10}\lambda_\mathrm{obs}$')

m = dlam > 0
axes[1].hist(dlam[m], bins=np.linspace(0, 40, 60), density=True, alpha=0.6)
axes[1].set_yscale('log')
axes[1].set_xlabel(r'$\lambda_\mathrm{obs}-\lambda_\mathrm{true}$')
axes[1].set_ylabel('p (log)')
fig.tight_layout()

## 5. Output 1 — $N(\lambda_\mathrm{obs}, z)$

In [ ]:
def compute_number_counts(x, z, xbins, zmin_list, zmax_list):
    counts = []
    for xl, xh in zip(xbins[:-1], xbins[1:]):
        xmask = (x >= xl) & (x < xh)
        row = []
        for zl, zh in zip(zmin_list, zmax_list):
            row.append(int(np.count_nonzero(xmask & (z >= zl) & (z < zh))))
        counts.append(row)
    return np.array(counts)

nc_obs = compute_number_counts(np.asarray(mock['lambda_obs']),
                                np.asarray(mock['redshift']),
                                LBDBINS, ZMIN_LIST, ZMAX_LIST)
print('N(lambda_obs, z):')
print(nc_obs)

## 6. Per-halo shear $\gamma_\mathrm{mock}$

Same recipe as `createDataVector.ipynb`: $\gamma = \Delta\Sigma\cdot\Sigma_\mathrm{crit}^{-1}(z_\mathrm{lens})$,
with $\beta_\mathrm{eff}$ tables from `mock_boost_factor_1d`.

In [ ]:
bTable  = np.load(floc.mock_boost_factor_1d)
betaEff = bTable['betaEff']; zlens = bTable['zlens']

const = 6.01e-19 * 1e6           # pc / Msun (4 pi G / c^2)
dlens = cosmo0.comoving_distance(zlens).value * 1e6   # pc
sigma_crit_vec = const * dlens[:, np.newaxis] * betaEff
sigma_crit_vec_z = sigma_crit_vec[:, -1]

redshift_sel = np.asarray(mock['redshift'])
deltaSigma   = np.asarray(mock['DeltaSigma'])
sigma_crit   = np.interp(redshift_sel, zlens, sigma_crit_vec_z)
shear        = deltaSigma * sigma_crit[:, np.newaxis]
print('shear array shape:', shear.shape)

## 7. $B_\mathrm{sel}$ per halo (one-halo limit)

$w_i = 1 + (\lambda_\mathrm{obs,i}-\lambda_\mathrm{true,i} -
\overline{\Delta_\mathrm{prj}})/\overline{\Delta_\mathrm{prj}}$ with
$\overline{\Delta_\mathrm{prj}} \equiv f_\mathrm{prj}/\tau$, the mean
projection boost at $(\lambda_\mathrm{obs}, z)$. The population mean of
$w_i$ over $P(\lambda_\mathrm{obs}|\lambda_\mathrm{true},z)$ is unity.

In [ ]:
w_bsel = b_sel_one_halo(np.asarray(mock['lambda_obs']),
                        np.asarray(mock['lambda_true']),
                        redshift_sel, PRJ_INTERP)
mock['B_sel'] = w_bsel
print('<B_sel> all       = %.3f' % w_bsel.mean())
print('<B_sel> lobs>=20  = %.3f' % w_bsel[mock['lambda_obs'] >= 20].mean())

## 8. Output 2 — $B_\mathrm{sel}$-weighted stacked $\gamma_\mathrm{obs}(R)$

$\gamma_\mathrm{obs}(R,\lambda_\mathrm{bin},z_\mathrm{bin}) =
\sum_i w_i\,\gamma_{\mathrm{mock},i}(R) / \sum_i w_i$, sum over halos
with $(\lambda_\mathrm{obs,i}, z_i)$ in the bin.

In [ ]:
class BSelStackingProfiles(object):
    """Weighted average of per-halo profile over (lambda_obs, z) bins."""
    def __init__(self, lam_bins, z_min, z_max):
        self.lam_bins = lam_bins; self.z_min = z_min; self.z_max = z_max
        self.nl = lam_bins.size - 1; self.nz = z_min.size

    def run(self, lobs, zcl, profile, weights):
        out_mean   = np.zeros((self.nl, self.nz, profile.shape[1]))
        out_wstack = np.zeros_like(out_mean)
        for i, (ll, lh) in enumerate(zip(self.lam_bins[:-1], self.lam_bins[1:])):
            for j, (zl, zh) in enumerate(zip(self.z_min, self.z_max)):
                m = (lobs >= ll) & (lobs < lh) & (zcl >= zl) & (zcl < zh)
                if not m.any():
                    continue
                p = profile[m]; w = weights[m]
                out_mean[i, j] = p.mean(axis=0)
                ws = w.sum()
                if ws > 0:
                    out_wstack[i, j] = (p * w[:, None]).sum(axis=0) / ws
        return out_mean, out_wstack

stacker  = BSelStackingProfiles(LBDBINS, ZMIN_LIST, ZMAX_LIST)
lobs_arr = np.asarray(mock['lambda_obs'])

gt_unw, gt_bsel = stacker.run(lobs_arr, redshift_sel, shear,      w_bsel)
ds_unw, ds_bsel = stacker.run(lobs_arr, redshift_sel, deltaSigma, w_bsel)
print('shapes:', gt_unw.shape, '(nl, nz, nr)')

In [ ]:
# Regression: weights = 1 should recover the unweighted stack.
_, gt_ones = stacker.run(lobs_arr, redshift_sel, shear, np.ones_like(w_bsel))
assert np.allclose(gt_ones, gt_unw)
print('OK: w=1 stack matches unweighted mean.')

In [ ]:
import radial_bins_phys_mpc as rbp
radii = rbp.rp_phys_mpc

fig, axes = plt.subplots(1, stacker.nl, figsize=(stacker.nl * 3.6 + 1, 4.0),
                          sharex=True, sharey=True,
                          gridspec_kw={'wspace': 0})
for i in range(stacker.nl):
    ax = axes[i]
    ax.loglog()
    for j in range(stacker.nz):
        ax.plot(radii, np.abs(gt_unw[i, j]),  '--', alpha=0.6,
                label=('unw  zb%i' % j)   if i == 0 else None)
        ax.plot(radii, np.abs(gt_bsel[i, j]),  '-',
                label=('B_sel zb%i' % j)  if i == 0 else None)
    ax.set_xlabel(r'R [Mpc]')
    ax.set_title(r'$%i<\lambda_\mathrm{obs}<%i$' %
                 (LBDBINS[i], LBDBINS[i + 1]))
axes[0].set_ylabel(r'$\gamma_t$')
axes[0].legend(fontsize=9, loc=1)
fig.tight_layout()

## 9. Save the data vector

In [ ]:
import h5py
out_fname = os.path.join(os.path.dirname(floc.mock_fname),
                          'dataVec_mock_buzzard_xin_v0.hdf5')
print('Save file:', out_fname)

with h5py.File(out_fname, 'w') as f:
    g = f.create_group('NC')
    g.create_dataset('vec',       data=nc_obs.flatten(), dtype='i')
    g.create_dataset('lbd_edges', data=LBDBINS)
    g.create_dataset('z_min',     data=ZMIN_LIST)
    g.create_dataset('z_max',     data=ZMAX_LIST)

    g = f.create_group('GT')
    g.create_dataset('gt_bsel', data=gt_bsel)
    g.create_dataset('gt_unw',  data=gt_unw)
    g.create_dataset('ds_bsel', data=ds_bsel)
    g.create_dataset('ds_unw',  data=ds_unw)
    g.create_dataset('radii',   data=radii)

    g = f.create_group('params')
    for name, val in [('M_min', M_MIN), ('alpha', ALPHA), ('M_1', M_1),
                       ('sigm_intr', SIGM_INTR), ('epsi', EPSI),
                       ('pivot_z0', PIVOT_Z0)]:
        g.attrs[name] = val
    g.attrs['prj_file']      = os.path.basename(PRJ_FILE)
    g.attrs['prj_reduction'] = 'posterior-mean'

print('done.')